# Model predictions

In [1]:
cd ../tcrsat

/Users/yang.an/PhD/Straub_An_et_al_2026_The-total-epitope-specific-T-cell-receptor-solution-space/TCRprediction/tcrsat


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
import numpy as np
import os
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

import sys
sys.path.append('..')

from tcrsat.models.predictor import Predictor
from tcrsat.data import TCRSatDataset


/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/lightning_fabric/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)


In [4]:
subset = 1000  # set to False for full dataset or lower for faster testing purposes

seq_data = pd.read_excel(
    '../../tcr_data/Supplementary table 2 TCR_data_busch_lab_ova_gp33_m45.xlsx',
    header=4, index_col=0,
)
seq_data['specificity'] = seq_data['annotated_specificity']
if subset:
    seq_data = seq_data.sample(subset, random_state=42)

seq_data = seq_data.rename(columns=lambda c: c + '_reported' if c.startswith('prediction_') else c)


In [5]:
datas = []
all_metric_results = []
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_path = 'saved_models'
criterion = 'auc'
mod = 'full'

for epi in ['siinfekl_reactive', 'GP33', 'M45']:
    for i in range(5):
        split = f'{i}_group_stratified'

        ckpt_path = f'../saved_models/{epi}/{epi}_{split}.ckpt'           
        model = Predictor.load_from_checkpoint(ckpt_path, map_location=torch.device(device))

        model.eval()
        model = model.to(device)

        data_config = model.data_config.copy()
        val_dataset = TCRSatDataset(config=data_config, split=None, tokenizer=model.tokenizer, data=seq_data)
        val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=128, shuffle=False)

        outputs = defaultdict(list)
        with torch.no_grad():
            for batch in tqdm(val_dataloader):
                batch = [item.to(device) for item in batch]
                x, attention_mask, y = batch
                y_hat_logits = model(x, attention_mask)
                y_hat = model.transform_predict(y_hat_logits)

                outputs['y_hat'].append(y_hat.detach())
                outputs['y'].append(y.detach())

            y_hat = torch.cat(outputs['y_hat'])
            y = torch.cat(outputs['y'])

            y_hat = y_hat.cpu()
            y = y.cpu()
            seq_data[f'prediction_{epi}_{i}'] = y_hat

    # Ensemble evaluation
    seq_data[f'prediction_{epi}_ensemble'] = seq_data[[f'prediction_{epi}_{i}' for i in range(5)]].mean(axis=1)


/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at facebook/esm2_t12_35M_UR50D were not used when initializing EsmModel: ['lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mod

Using LoRA
trainable params: 368640 || all params: 34361521 || trainable%: 1.072827946120313


100%|██████████| 8/8 [02:48<00:00, 21.04s/it]
/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at facebook/esm2_t12_35M_UR50D were not used when initializing EsmModel: ['lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification 

Using LoRA
trainable params: 368640 || all params: 34361521 || trainable%: 1.072827946120313


100%|██████████| 8/8 [03:17<00:00, 24.74s/it]
/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at facebook/esm2_t12_35M_UR50D were not used when initializing EsmModel: ['lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification 

...

100%|██████████| 8/8 [06:14<00:00, 46.87s/it]
/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at facebook/esm2_t30_150M_UR50D were not used when initializing EsmModel: ['lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification

Using LoRA
trainable params: 2457600 || all params: 151253081 || trainable%: 1.6248264060154913


100%|██████████| 8/8 [09:31<00:00, 71.40s/it]
/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at facebook/esm2_t30_150M_UR50D were not used when initializing EsmModel: ['lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing EsmModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification

Using LoRA
trainable params: 2457600 || all params: 151253081 || trainable%: 1.6248264060154913


100%|██████████| 8/8 [08:19<00:00, 62.43s/it]


In [6]:
# Compare with reported results

TOL = 1e-4

rows = []
for epi in ['siinfekl_reactive', 'GP33', 'M45']:
    for i in range(5):
        new, old = f'prediction_{epi}_{i}', f'prediction_{epi}_split_{i}_reported'
        if new not in seq_data or old not in seq_data:
            continue
        diff = (seq_data[new] - seq_data[old]).abs()
        rows.append({'column': new, 'max_abs_diff': diff.max(),
                     'n_off': int((diff > TOL).sum()), 'ok': diff.max() <= TOL})

comparison = pd.DataFrame(rows)
display(comparison)
assert not comparison.empty, 'no column pairs matched'
assert comparison['ok'].all(), comparison[~comparison['ok']]


[autoreload of seaborn.utils failed: Traceback (most recent call last):
  File "/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 619, in _exec
  File "<frozen importlib._bootstrap_external>", line 883, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/Users/yang.an/.local/share/mamba/envs/tcrPrediction/lib/python3.10/site-packages/seaborn/utils.py", line 17, in <module>
    from seaborn._core.typing import deprecated
ImportError: can

,column,max_abs_diff,n_off,ok
0,prediction_siinfekl_reactive_0,0.000009,0,True
1,prediction_siinfekl_reactive_1,0.000013,0,True
2,prediction_siinfekl_reactive_2,0.000013,0,True
3,prediction_siinfekl_reactive_3,0.000009,0,True
4,prediction_siinfekl_reactive_4,0.000011,0,True
5,prediction_GP33_0,0.000011,0,True
6,prediction_GP33_1,0.000007,0,True
7,prediction_GP33_2,0.000013,0,True
8,prediction_GP33_3,0.000005,0,True
9,prediction_GP33_4,0.000018,0,True
